In [ ]:
import os
import sys

# Auto-detect Colab and fetch data
if 'google.colab' in sys.modules:
    if not os.path.exists("LLMs-and-GenAI-Assignment"):
        !git clone https://github.com/nelsunnel/LLMs-and-GenAI-Assignment.git
    
    # Change directory so all relative paths (Datasets/...) work properly
    os.chdir("LLMs-and-GenAI-Assignment")
    print("Running in Colab: clonded repository and navigated to project root.")
else:
    print("Running locally.")
    
!pip install torchvision torch matplotlib pandas seaborn numpy

In [ ]:
import os
import glob
import xml.etree.ElementTree as ET
from PIL import Image
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms.functional as F

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

# The XMLs use obj1, obj2, obj3, obj4. We map them logically to the 4 classes.
# 0 is always reserved for background in torchvision models
CLASS_MAPPING = {
    'background': 0,
    'obj1': 1,
    'obj2': 2,
    'obj3': 3,
    'obj4': 4
}

FRIENDLY_NAMES = {
    0: 'Background',
    1: 'Water Bottle',
    2: 'Milk Bottle',
    3: 'Tetra Pack',
    4: 'Can'
}

COLOR_MAP = {
    1: '#4C72B0', # Blue
    2: '#55A868', # Green
    3: '#C44E52', # Red
    4: '#8172B2'  # Purple
}

In [ ]:
class ObjectDetectionDataset(Dataset):
    def __init__(self, data_dir, image_files):
        self.data_dir = data_dir
        self.image_files = image_files
        
    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.data_dir, "images", img_name)
        
        # Determine matching XML annotation
        xml_name = img_name.replace(".jpg", ".xml")
        xml_path = os.path.join(self.data_dir, "annotations", xml_name)
        
        # Load image
        img = Image.open(img_path).convert("RGB")
        img_tensor = F.to_tensor(img)
        
        # Parse XML
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        boxes = []
        labels = []
        for obj in root.findall('object'):
            name = obj.find('name').text
            if name not in CLASS_MAPPING:
                continue
                
            label = CLASS_MAPPING[name]
            bndbox = obj.find('bndbox')
            xmin = float(bndbox.find('xmin').text)
            ymin = float(bndbox.find('ymin').text)
            xmax = float(bndbox.find('xmax').text)
            ymax = float(bndbox.find('ymax').text)
            
            # Faster RCNN expects boxes in [xmin, ymin, xmax, ymax]
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(label)
            
        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        else:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
            
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.empty((0,))
        iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
        
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = torch.tensor([idx])
        target["area"] = area
        target["iscrowd"] = iscrowd
        
        return img_tensor, target

# The dataset needs to be split: first 100 for train, remaining 28 for test.
# We must sort files numerically (1.jpg, 2.jpg, ... 128.jpg) to ensure sequential splitting.
all_images = os.listdir('Datasets/dataset2/images')
valid_images = [f for f in all_images if f.endswith('.jpg')]
valid_images.sort(key=lambda x: int(os.path.splitext(x)[0]))

train_images = valid_images[:100]
test_images = valid_images[100:]

dataset2_dir = 'Datasets/dataset2'

train_dataset = ObjectDetectionDataset(dataset2_dir, train_images)
test_dataset = ObjectDetectionDataset(dataset2_dir, test_images)

def collate_fn(batch):
    return tuple(zip(*batch))

# Using small batch sizes as Faster RCNN consumes heavy VRAM
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f"Train Dataset Size: {len(train_dataset)}")
print(f"Test Dataset Size: {len(test_dataset)}")

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

# Number of classes is 5 (1 background + 4 actual classes)
num_classes = 5

# Load a pre-trained model for detection
model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)

# Get number of input features for the classifier
in_features = model.roi_heads.box_predictor.cls_score.in_features
# Replace the pre-trained head with a new one
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

model = model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=1e-4, weight_decay=0.0005)

# Optional learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

In [ ]:
num_epochs = 5

print(f"Starting Training for {num_epochs} Epochs...")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for images, targets in train_loader:
        # Move images and targets to device
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        epoch_loss += losses.item()
        
    lr_scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.4f}")

In [ ]:
def calculate_iou(boxA, boxB):
    # Determine the (x, y)-coordinates of the intersection rectangle
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)

    boxAArea = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
    boxBArea = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)

    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

print("Starting Evaluation on Test Set...")
model.eval()

iou_threshold = 0.5

# We will collect True Positives (TP), False Positives (FP) and total Ground Truths (Num_GT) per class
class_tp = {1: 0, 2: 0, 3: 0, 4: 0}
class_fp = {1: 0, 2: 0, 3: 0, 4: 0}
class_num_gt = {1: 0, 2: 0, 3: 0, 4: 0}

with torch.no_grad():
    for images, targets in test_loader:
        images = list(img.to(device) for img in images)
        
        outputs = model(images)
        
        for i, output in enumerate(outputs):
            target = targets[i]
            
            pred_boxes = output['boxes'].cpu().numpy()
            pred_scores = output['scores'].cpu().numpy()
            pred_labels = output['labels'].cpu().numpy()
            
            gt_boxes = target['boxes'].cpu().numpy()
            gt_labels = target['labels'].cpu().numpy()
            
            # Count ground truths per class
            for lbl in gt_labels:
                class_num_gt[lbl] += 1
            
            # Array to keep track of matched GT boxes
            matched_gt = np.zeros(len(gt_boxes))
            
            # Sort predictions by score descending
            sorted_indices = np.argsort(pred_scores)[::-1]
            pred_boxes = pred_boxes[sorted_indices]
            pred_labels = pred_labels[sorted_indices]
            
            for p_idx, p_box in enumerate(pred_boxes):
                p_label = pred_labels[p_idx]
                
                # Find matching ground truth
                best_iou = 0
                best_gt_idx = -1
                
                for g_idx, g_box in enumerate(gt_boxes):
                    g_label = gt_labels[g_idx]
                    
                    if p_label == g_label and matched_gt[g_idx] == 0:
                        iou = calculate_iou(p_box, g_box)
                        if iou > best_iou:
                            best_iou = iou
                            best_gt_idx = g_idx
                            
                if best_iou >= iou_threshold:
                    class_tp[p_label] += 1
                    matched_gt[best_gt_idx] = 1 # Mark matched
                else:
                    class_fp[p_label] += 1

print("\nEvaluation Metrics (IoU >= 0.5):")
for cls_id in range(1, 5):
    tp = class_tp[cls_id]
    fp = class_fp[cls_id]
    num_gt = class_num_gt[cls_id]
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / num_gt if num_gt > 0 else 0
    
    print(f"Class: {FRIENDLY_NAMES[cls_id]:<15} | Precision: {precision:.4f} | Recall: {recall:.4f}")

In [ ]:
def plot_predictions(image_tensor, output_dict, score_threshold=0.5):
    """
    Renders the image with bounding boxes using seaborn-style colors.
    """
    img = image_tensor.permute(1, 2, 0).numpy()
    
    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(img)
    ax.axis('off')
    
    boxes = output_dict['boxes'].cpu().numpy()
    scores = output_dict['scores'].cpu().numpy()
    labels = output_dict['labels'].cpu().numpy()
    
    for idx, box in enumerate(boxes):
        if scores[idx] < score_threshold:
            continue
            
        xmin, ymin, xmax, ymax = box
        width = xmax - xmin
        height = ymax - ymin
        
        lbl = labels[idx]
        color = COLOR_MAP.get(lbl, 'white')
        name = FRIENDLY_NAMES.get(lbl, 'Unknown')
        score = scores[idx]
        
        # Draw bounding box
        rect = patches.Rectangle((xmin, ymin), width, height, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        # Add label
        label_text = f"{name} {score:.2f}"
        ax.text(xmin, ymin - 5, label_text, color='white', fontsize=10, weight='bold',
                bbox=dict(facecolor=color, edgecolor='none', alpha=0.8, boxstyle='round,pad=0.2'))
        
    plt.tight_layout()
    plt.show()

print("Visualizing predictions on a few Test Set examples...")
model.eval()

# To pick a few samples visually nicely
eval_indices = [0, 5, 10, 15]

with torch.no_grad():
    for idx in eval_indices:
        if idx >= len(test_dataset):
            continue
        image, target = test_dataset[idx]
        output = model([image.to(device)])[0]
        
        plot_predictions(image, output, score_threshold=0.6)